# FRED Data Preprocessing
Cleaning the raw risk-free rate and CPI series: filling gaps, converting units, and aligning to trading dates.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
RAW_DATA_PATH = Path('../data/raw')
PROCESSED_DATA_PATH = Path('../data/processed')

## Load raw data

In [ ]:
rf = pd.read_csv(RAW_DATA_PATH / 'fred_risk_free_rate_raw.csv', index_col='date', parse_dates=True)
cpi = pd.read_csv(RAW_DATA_PATH / 'fred_cpi_raw.csv', index_col='date', parse_dates=True)

print("Risk-free rate shape:", rf.shape)
print("CPI shape:", cpi.shape)

## Clean the risk-free rate series
FRED's Treasury series has gaps on weekends/holidays. We reindex to every calendar day and forward-fill,
since a rate stays constant until the next published update.

In [ ]:
rf_filled = rf.asfreq('D').ffill()

print("Missing values after ffill:", rf_filled['risk_free_rate_pct'].isna().sum())
rf_filled.head(10)

## Convert annualized percentage to decimal
FRED reports this as an annualized percent (e.g. 5.25 = 5.25%). Convert to decimal for use in Sharpe ratio math.

In [ ]:
rf_filled['risk_free_rate_decimal'] = rf_filled['risk_free_rate_pct'] / 100
rf_filled.head()

## Align to trading dates
Replace this with the actual trading date index from the price data notebook once available.
For now this uses US business days as a placeholder.

In [ ]:
# TODO: replace with actual trading dates from price data, e.g.:
# trading_dates = pd.read_csv(PROCESSED_DATA_PATH / 'prices_processed.csv', index_col='date', parse_dates=True).index

trading_dates = pd.bdate_range(start=rf_filled.index.min(), end=rf_filled.index.max())
rf_aligned = rf_filled.reindex(trading_dates).ffill()
rf_aligned.index.name = 'date'

print(f"Aligned to {len(rf_aligned)} trading days")
rf_aligned.head()

## Clean CPI (monthly -> forward-filled to daily, for merging with daily context if needed)

In [ ]:
cpi_filled = cpi.asfreq('D').ffill()
cpi_filled['cpi_pct_change'] = cpi_filled['cpi_index'].pct_change()
cpi_filled.head()

## Save processed data

In [ ]:
rf_aligned.to_csv(PROCESSED_DATA_PATH / 'fred_risk_free_rate_processed.csv')
cpi_filled.to_csv(PROCESSED_DATA_PATH / 'fred_cpi_processed.csv')

print("Saved processed FRED data to", PROCESSED_DATA_PATH)